# Deep Learning Assignment 2
## Cross-Subject vs. Intra-Subject Task Classification on Multi-Channel MEG Data

In [ ]:
import copy
import os
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import resample
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, Dataset

sys.path.append(str(Path('..').resolve()))
from src.preprocessing import downsample_matrix, preprocess_matrix, apply_meg_augmentation

In [ ]:
data_root = Path('..') / 'data' / 'raw' / 'Final Project data'
h5_files = sorted(data_root.rglob('*.h5'))
task_prefixes = ['rest_', 'task_motor_', 'task_story_math_', 'task_working_memory_']

if not h5_files:
    raise FileNotFoundError(f'No .h5 files found under {data_root}')

def get_dataset_name(file_name_with_dir: Path) -> str:
    file_name_without_dir = file_name_with_dir.as_posix().split('/')[-1]
    temp = file_name_without_dir.split('_')[:-1]
    dataset_name = "_".join(temp)
    return dataset_name

def get_first_h5_file_path_foreach_task(h5_files, task_prefixes):
    for prefix in task_prefixes:
        for file_path in h5_files:
            if file_path.name.startswith(prefix):
                return file_path
    raise ValueError(f"No file found with prefixes: {task_prefixes}")

def get_amount_of_files_per_task(h5_files, task_prefixes):
    counts = {prefix: 0 for prefix in task_prefixes}
    for file_path in h5_files:
        for prefix in task_prefixes:
            if file_path.name.startswith(prefix):
                counts[prefix] += 1
                break
    return counts

filename_paths = [get_first_h5_file_path_foreach_task(h5_files, [prefix]) for prefix in task_prefixes]

for filename_path in filename_paths:
    print(f'Loading: {filename_path}')
    with h5py.File(filename_path, 'r') as f:
        dataset_name = get_dataset_name(filename_path)
        obj = f.get(dataset_name)
        if obj is None:
            raise ValueError(f"Dataset '{dataset_name}' not found in {filename_path}")
        if not isinstance(obj, h5py.Dataset):
            raise ValueError(f"Expected h5py.Dataset, got {type(obj)}")
        dataset: h5py.Dataset = obj
        matrix: np.ndarray = dataset[()]
        print(type(matrix))
        print(matrix.shape)

get_amount_of_files_per_task(h5_files, task_prefixes)

In [ ]:
target_downsample_rate = 500

task_data = {}

for file_path in filename_paths:
    dataset_name = get_dataset_name(file_path)
    prefix = next((p for p in task_prefixes if file_path.name.startswith(p)), "unknown")
    
    with h5py.File(file_path, 'r') as f:
        obj = f.get(dataset_name)
        if obj is None:
            raise ValueError(f"Dataset '{dataset_name}' not found in {file_path}")
        if not isinstance(obj, h5py.Dataset):
            raise TypeError(f"Expected h5py.Dataset, got {type(obj)}")
        raw_matrix = obj[()]
        
    preprocessed_matrix = preprocess_matrix(raw_matrix)
    downsampled_matrix = downsample_matrix(preprocessed_matrix, downsample_rate=target_downsample_rate)
    downsample_factor = preprocessed_matrix.shape[1] / downsampled_matrix.shape[1]
    
    task_data[prefix] = {
        'raw': raw_matrix,
        'preprocessed': preprocessed_matrix,
        'downsampled': downsampled_matrix,
        'factor': downsample_factor,
        'file_name': file_path.name
    }
    
    print(f"Task Prefix: {prefix} ({file_path.name})")
    print(f"Original/Preprocessed Matrix Shape: {preprocessed_matrix.shape}")
    print(f"          Downsampled Matrix Shape: {downsampled_matrix.shape}")
    print(f"               Downsampling Factor: {downsample_factor:.2f}x fewer time points\n")

In [ ]:
channel_to_visualize = 0
raw_time_window_length = 1000

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for idx, (prefix, data) in enumerate(task_data.items()):
    ax = axes[idx]
    
    preprocessed_matrix = data['preprocessed']
    downsampled_matrix = data['downsampled']
    downsample_factor = data['factor']
    
    time_window_raw = slice(0, raw_time_window_length)
    time_window_ds = slice(0, int(raw_time_window_length / downsample_factor))

    x_raw = np.arange(0, raw_time_window_length)
    x_ds = np.arange(0, int(raw_time_window_length / downsample_factor)) * downsample_factor

    ax.plot(x_raw, preprocessed_matrix[channel_to_visualize, time_window_raw], 
             label='Before Downsampling', color='#1f77b4', alpha=0.4, linewidth=1.5)

    ax.plot(x_ds, downsampled_matrix[channel_to_visualize, time_window_ds], 
             label=f'After Downsampling (Factor: {downsample_factor:.1f})', 
             color='#ff7f0e', alpha=1, linewidth=1.5, linestyle='-')

    ax.set_title(f'[{prefix}] Channel {channel_to_visualize} Signal - Comparison Overlay', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Steps', fontsize=10)
    ax.set_ylabel('Amplitude', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/CNN-GRU/preprocessing_meg_signal_overlay_comparison.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8))
axes_flat = axes.flat 

for i, filename_path in enumerate(filename_paths):
    if i >= len(axes_flat):
        break
        
    dataset_name = get_dataset_name(filename_path)
    prefix = next((p for p in task_prefixes if filename_path.name.startswith(p)), "unknown")
    data = task_data[prefix]
    
    ax = axes_flat[i]
    
    ax.imshow(data['raw'][:downsampled_matrix.shape[0], :], aspect='auto', cmap='viridis')
    ax.set_title(f'Raw MEG Signal - {prefix.replace("_", " ")}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time Steps', fontsize=10)
    ax.set_ylabel('Channels', fontsize=10)
    ax.grid(False)

plt.tight_layout()
plt.savefig('../reports/figures/CNN-GRU/visualization_of_the_data.png', dpi=300)
plt.show()

In [ ]:
%%script Skipped_Cell

num_channels_to_plot = 248 
raw_heatmap_time_length = 800 

fig, axes = plt.subplots(4, 2, figsize=(18, 20))

for idx, (prefix, data) in enumerate(task_data.items()):
    ax1 = axes[idx, 0]
    ax2 = axes[idx, 1]
    
    preprocessed_matrix = data['preprocessed']
    downsampled_matrix = data['downsampled']
    downsample_factor = data['factor']

    heatmap_time_raw = preprocessed_matrix[:num_channels_to_plot, :raw_heatmap_time_length]
    heatmap_time_ds = downsampled_matrix[:num_channels_to_plot, :int(raw_heatmap_time_length / downsample_factor)]

    sns.heatmap(heatmap_time_raw, cmap='viridis', ax=ax1, 
                cbar_kws={'label': 'Signal Intensity'})
    ax1.set_title(f'[{prefix}] Before Downsampling', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Time Steps (Original)', fontsize=10)
    ax1.set_ylabel('Channels', fontsize=10)

    sns.heatmap(heatmap_time_ds, cmap='viridis', ax=ax2, 
                cbar_kws={'label': 'Signal Intensity'})
    ax2.set_title(f'[{prefix}] After Downsampling ({target_downsample_rate}Hz)', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Time Steps (Downsampled)', fontsize=10)
    ax2.set_ylabel('Channels', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/CNN-GRU/preprocessing_meg_heatmap_comparison.png', dpi=300)
plt.show()

In [ ]:
%%script Skipped_Cell

for idx, (prefix, data) in enumerate(task_data.items()):
  matrix_data = task_data[prefix]

  preprocessed_matrix = data['preprocessed']
  downsampled_matrix = data['downsampled']
  downsample_factor = data['factor']

  num_channels = min(preprocessed_matrix.shape[0], downsampled_matrix.shape[0])
  
  num_time_steps = int(preprocessed_matrix.shape[1] / downsample_factor)

  num_channels = min(num_channels, preprocessed_matrix.shape[0], downsampled_matrix.shape[0])
  num_time_steps = min(num_time_steps, preprocessed_matrix.shape[1], downsampled_matrix.shape[1])

  preprocessed_slice = preprocessed_matrix[:num_channels, :num_time_steps]
  downsampled_slice = downsampled_matrix[:num_channels, :num_time_steps]

  fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

  sns.heatmap(preprocessed_slice, ax=axes[0], cmap='viridis', cbar=True, 
              cbar_kws={'label': 'Activity Intensity'})
  axes[0].set_title(f"Preprocessed Neuron Activity ({prefix})\nGrid: {preprocessed_slice.shape[0]} Channels × {preprocessed_slice.shape[1]} Time Steps")
  axes[0].set_xlabel("Time Steps")
  axes[0].set_ylabel("Channels / Neurons")

  sns.heatmap(downsampled_slice, ax=axes[1], cmap='viridis', cbar=True, 
              cbar_kws={'label': 'Activity Intensity'})
  axes[1].set_title(f"Downsampled Neuron Activity ({prefix})\nGrid: {downsampled_slice.shape[0]} Channels × {downsampled_slice.shape[1]} Time Steps")
  axes[1].set_xlabel("Time Steps")

plt.tight_layout()
plt.show()

In [ ]:
label_map = {
    "rest": 0,
    "task_motor": 1,
    "task_story_math": 2,
    "task_working_memory": 3
}

class MEGDataset(Dataset):
    def __init__(self, file_paths, persist_dir=None, downsample_rate=250, augment=False):
        self.file_paths = file_paths
        self.persist_dir = persist_dir
        self.downsample_rate = downsample_rate
        self.augment = augment
        if persist_dir:
            Path(persist_dir).mkdir(parents=True, exist_ok=True)
            
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        filename = file_path.name

        label_str = next((k for k in label_map.keys() if filename.startswith(k)), None)
        if label_str is None:
            raise ValueError(f"Unknown label in filename: {filename}")

        label = label_map[label_str]

        subject_id = filename.split('_')[-2]
        
        if self.persist_dir:
            save_path = Path(self.persist_dir) / f"{filename}.npy"
            if save_path.exists():
                matrix = np.load(save_path)
            else:
                with h5py.File(file_path, 'r') as f:
                    dataset_name = "_".join(filename.split('_')[:-1])
                    obj = f.get(dataset_name)
                    if obj is None:
                        raise ValueError(f"Dataset '{dataset_name}' not found in {file_path}")
                    matrix = obj[()]
                
                matrix = preprocess_matrix(matrix)
                matrix = downsample_matrix(matrix, downsample_rate=self.downsample_rate)
                np.save(save_path, matrix)
        else:
            with h5py.File(file_path, 'r') as f:
                dataset_name = "_".join(filename.split('_')[:-1])
                obj = f.get(dataset_name)
                if obj is None:
                    raise ValueError(f"Dataset '{dataset_name}' not found in {file_path}")
                matrix = obj[()]
            
            matrix = preprocess_matrix(matrix)
            matrix = downsample_matrix(matrix, downsample_rate=self.downsample_rate)

        if self.augment:
            matrix = apply_meg_augmentation(matrix)
            
        return torch.tensor(matrix, dtype=torch.float32), torch.tensor(label, dtype=torch.long), subject_id

In [ ]:
data_root = Path('..') / 'data' / 'raw' / 'Final Project data'
processed_dir = Path('..') / 'data' / 'processed'

all_intra_files = sorted(list((data_root / 'Intra').rglob('*.h5')))

# Extract labels from filenames for stratification
task_prefixes_for_label = ['rest_', 'task_motor_', 'task_story_math_', 'task_working_memory_']
file_labels = []
for f in all_intra_files:
    label = next((p for p in task_prefixes_for_label if f.name.startswith(p)), 'unknown')
    file_labels.append(label)


split_idx = int(0.8 * len(all_intra_files))
train_files, test_files = train_test_split(
    all_intra_files,
    test_size=0.2,
    stratify=file_labels,   # ← ensures all 4 classes appear in both splits
    random_state=42
)

intra_train_dataset = MEGDataset(train_files, persist_dir=processed_dir)
intra_test_dataset = MEGDataset(test_files, persist_dir=processed_dir)

batch_size = 16
intra_train_loader = DataLoader(intra_train_dataset, batch_size=batch_size, shuffle=True)
intra_test_loader = DataLoader(intra_test_dataset, batch_size=batch_size, shuffle=False)

print(f"Intra-Subject Block Split: {len(train_files)} train files, {len(test_files)} test files")
print(f"Total samples: {len(intra_train_dataset)} train samples, {len(intra_test_dataset)} test samples")

In [ ]:
cross_train_files = list((data_root / 'Cross' / 'train').rglob('*.h5'))
cross_test_files = []
for i in [1, 2, 3]:
    cross_test_files.extend((data_root / 'Cross' / f'test{i}').rglob('*.h5'))

cross_train_dataset = MEGDataset(cross_train_files, persist_dir=processed_dir)
cross_test_dataset = MEGDataset(cross_test_files, persist_dir=processed_dir)

cross_train_loader = DataLoader(cross_train_dataset, batch_size=batch_size, shuffle=True)
cross_test_loader = DataLoader(cross_test_dataset, batch_size=batch_size, shuffle=False)

print(f"Cross-Subject Block Split: {len(cross_train_files)} train files, {len(cross_test_files)} test files")
print(f"Total samples: {len(cross_train_dataset)} train samples, {len(cross_test_dataset)} test samples")

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        # x: [batch, seq_len, hidden_dim]
        attn_weights = self.attn(x) # [batch, seq_len, 1]
        attn_weights = F.softmax(attn_weights, dim=1)
        context = torch.sum(attn_weights * x, dim=1) # [batch, hidden_dim]
        return context

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

def grl(x, alpha):
    return GradientReversalLayer.apply(x, alpha)

class CNN_1D_GRU_Classifier(nn.Module):
    def __init__(self, in_channels=248, num_classes=4, dropout=0.5, hidden_dim=64, negative_slope=0.01):
        super().__init__()

        self.conv = nn.Sequential(
            
            nn.Conv1d(in_channels, 248, kernel_size=15, padding=15 // 2),
            nn.InstanceNorm1d(248),
            nn.LeakyReLU(negative_slope),
            nn.Dropout1d(0.2), 
            nn.MaxPool1d(2),   

            nn.Conv1d(248, 256, kernel_size=7, padding=7 // 2),
            nn.InstanceNorm1d(256),
            nn.LeakyReLU(negative_slope),
            nn.MaxPool1d(2)
        )

        self.gru = nn.GRU(
            input_size=256, 
            hidden_size=hidden_dim, 
            num_layers=1,
            batch_first=True, 
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 128),
            nn.LeakyReLU(negative_slope),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        
        x = self.conv(x) 

        x = x.permute(0, 2, 1) 

        _, h_n = self.gru(x)

        h_forward = h_n[0, :, :]
        h_backward = h_n[1, :, :]

        out = torch.cat((h_forward, h_backward), dim=1) 

        return self.fc(out)

def build_model_1D_GRU_Classifier(input_channels=248, num_classes=4):
    return CNN_1D_GRU_Classifier(in_channels=input_channels, num_classes=num_classes)

cnn_1d_gru_classifier = build_model_1D_GRU_Classifier()
print(cnn_1d_gru_classifier)

class DANN_Classifier(nn.Module):
    def __init__(self, feature_extractor, num_tasks=4, num_subjects=2, dropout=0.5, negative_slope=0.01):
        super().__init__()
        self.feature_extractor = feature_extractor

        if isinstance(feature_extractor, nn.Sequential):

            feature_dim = 256 
        elif hasattr(feature_extractor, 'gru'):
            
            feature_dim = 64
        else:
            feature_dim = 256

        self.task_classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(dropout),
            nn.Linear(128, num_tasks)
        )
        
        self.domain_classifier = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(dropout),
            nn.Linear(128, num_subjects)
        )

    def forward(self, x, alpha=1.0):
        
        if hasattr(self.feature_extractor, 'gru'):
            
            feat = self.feature_extractor.conv(x)
            feat = feat.permute(0, 2, 1)
            gru_out, _ = self.feature_extractor.gru(feat)
            features = self.feature_extractor.attention(gru_out)
        else:
            
            features = self.feature_extractor(x)
            features = torch.flatten(features, 1)

        task_out = self.task_classifier(features)

        reversed_features = grl(features, alpha)
        domain_out = self.domain_classifier(reversed_features)
        
        return task_out, domain_out

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs=50, learning_rate=1e-3, patience=10, augment=False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4) 
    
    
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-5)
    
    results = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': [], 'train_f1': [], 'test_f1': []}
    
    best_test_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        all_train_preds = []
        all_train_labels = []
        
        for batch in train_loader:
            inputs, labels, _ = batch 
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        epoch_f1 = f1_score(all_train_labels, all_train_preds, average='macro')
        
        model.eval()
        test_loss_sum, test_correct, test_total = 0.0, 0, 0
        all_test_preds = []
        all_test_labels = []
        with torch.no_grad():
            for batch in test_loader:
                inputs, labels, _ = batch
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss_sum += loss.item() * inputs.size(0)
                
                _, predicted = torch.max(outputs, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()
                
                all_test_preds.extend(predicted.cpu().numpy())
                all_test_labels.extend(labels.cpu().numpy())
                
        test_loss = test_loss_sum / test_total if test_total > 0 else 0
        test_acc = test_correct / test_total if test_total > 0 else 0
        test_f1 = f1_score(all_test_labels, all_test_preds, average='macro') if test_total > 0 else 0
        
        current_lr = optimizer.param_groups[0]['lr']

        if epoch % 10 == 0:
            print(f"Epoch {epoch+1:02d}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.4f} - Train F1: {epoch_f1:.4f} - Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f} - Test F1: {test_f1:.4f} - LR: {current_lr:.2e}")
        
        scheduler.step()
        
        results['train_loss'].append(epoch_loss)
        results['train_acc'].append(epoch_acc)
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)
        results['train_f1'].append(epoch_f1)
        results['test_f1'].append(test_f1)
        
        
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
                
    model.load_state_dict(best_model_wts)
    return results

In [ ]:

print("running intra-subject training for CNN_1D_GRU_Classifier")
intra_model_1D_GRU_Classifier = build_model_1D_GRU_Classifier()
intra_subject_results_1D_GRU_Classifier = train_model(intra_model_1D_GRU_Classifier, intra_train_loader, intra_test_loader)

In [ ]:
print("running cross-subject training for CNN_1D_GRU_Classifier")
cross_model_1D_GRU_Classifier = build_model_1D_GRU_Classifier()
cross_subject_results_1D_GRU_Classifier = train_model(cross_model_1D_GRU_Classifier, cross_train_loader, cross_test_loader)

In [ ]:
def plot_learning_curves(results_dict, title, model_name):
    epochs = range(1, len(results_dict['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, results_dict['train_loss'], label='Train Loss')
    ax1.set_title(f'{title} ({model_name}) - Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    
    ax2.plot(epochs, results_dict['train_acc'], label='Train Accuracy')
    ax2.plot(epochs, results_dict['test_acc'], label='Test Accuracy')
    ax2.set_title(f'{title} ({model_name}) - Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    
    Path('../reports/figures').mkdir(parents=True, exist_ok=True)
    filename = f"learning_curves_{title.replace(' ', '_').lower()}_{model_name.lower()}.png"
    plt.savefig(f"../reports/figures/CNN-GRU/{filename}")
    plt.show()

plot_learning_curves(intra_subject_results_1D_GRU_Classifier, 'Intra Subject', 'CNN1D_GRU')
plot_learning_curves(cross_subject_results_1D_GRU_Classifier, 'Cross Subject', 'CNN1D_GRU')

print(f"Intra-Subject CNN_1D_GRU Test Acc: {intra_subject_results_1D_GRU_Classifier['test_acc'][-1]:.4f}")
print(f"Cross-Subject CNN_1D_GRU Test Acc: {cross_subject_results_1D_GRU_Classifier['test_acc'][-1]:.4f}")
print(f"CNN_1D_GRU Gap: {intra_subject_results_1D_GRU_Classifier['test_acc'][-1] - cross_subject_results_1D_GRU_Classifier['test_acc'][-1]:.4f}")

In [ ]:
def plot_confusion_matrix(model, test_loader, title, model_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in test_loader:
            inputs, labels, _ = batch
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    cm = confusion_matrix(all_labels, all_preds, labels=list(label_map.values()))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=list(label_map.keys()), 
                yticklabels=list(label_map.keys()))
    plt.title(f'{title} ({model_name}) - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    
    filename = f"cm_{title.replace(' ', '_').lower()}_{model_name.lower()}.png"
    plt.savefig(f"../reports/figures/CNN-GRU/{filename}")
    
    plt.clf()
    return cm

confusion_matrices = {}
confusion_matrices['Intra Subject CNN1D_GRU'] = plot_confusion_matrix(intra_model_1D_GRU_Classifier, intra_test_loader, 'Intra Subject', 'CNN1D_GRU')
confusion_matrices['Cross Subject CNN1D_GRU'] = plot_confusion_matrix(cross_model_1D_GRU_Classifier, cross_test_loader, 'Cross Subject', 'CNN1D_GRU')


fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for idx, (title, cm) in enumerate(confusion_matrices.items()):
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=list(label_map.keys()), 
                yticklabels=list(label_map.keys()), ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.savefig("../reports/figures/CNN-GRU/cm_comparison_all_models.png")
plt.show()

In [ ]:
def evaluate_metrics(model, test_loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels, _ in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
                
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    try:
        roc_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    except:
        roc_auc = None  
        
    mae = mean_absolute_error(all_labels, all_preds)
    rmse = np.sqrt(mean_squared_error(all_labels, all_preds))
    
    return {
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "MAE": mae,
        "RMSE": rmse
    }

print("Training all 4 models on cross-subject data...")

models_to_test = {
    "CNN_1D_GRU_Classifier": CNN_1D_GRU_Classifier(in_channels=248, num_classes=4),
}

final_performance = []

for name, classifier in models_to_test.items():
    print(f"\n--- Training {name} ---")
    
    
    res_CNN_GRU = train_model(classifier, cross_train_loader, cross_test_loader)
    
    metrics = evaluate_metrics(classifier, cross_test_loader)
    metrics["Model"] = name
    metrics["Test Accuracy"] = res_CNN_GRU['test_acc'][-1]
    
    plot_learning_curves(res_CNN_GRU, f'{name} Cross Subject Training', model_name=name)
    plot_confusion_matrix(classifier, cross_test_loader, f'Cross Subject {name}', name)
    
    final_performance.append(metrics)
    
df_performance = pd.DataFrame(final_performance)
df_performance = df_performance[['Model', 'Test Accuracy', 'F1-Score', 'ROC-AUC', 'MAE', 'RMSE']]
df_performance.to_csv('../reports/model_performance_comparison.csv', index=False)
display(df_performance)

In [ ]:

np.random.seed(42)
torch.manual_seed(42)

def extract_features_and_labels(model, data_loader, device):
    """
    Extracts features and labels from the dataset using the specified model.
    If model is None, it flattens and extracts the preprocessed raw data.
    """
    all_features = []
    all_labels = []
    
    if model is not None:
        model.eval()
    
    with torch.no_grad():
        for inputs, labels, _ in data_loader:
            inputs = inputs.to(device)
            
            if model is not None:
                
                outputs = model(inputs)
                features = outputs.cpu().numpy()
            else:
                
                features = inputs.view(inputs.size(0), -1).cpu().numpy()
                
            all_features.append(features)
            all_labels.append(labels.numpy())
            
    return np.concatenate(all_features, axis=0), np.concatenate(all_labels, axis=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

active_model = cross_model_1D_GRU_Classifier  

all_files = sorted(data_root.rglob('*.h5'))
all_dataset = MEGDataset(all_files, persist_dir=processed_dir)
all_loader = DataLoader(all_dataset, batch_size=16, shuffle=False)

active_loader = all_loader  

features, labels = extract_features_and_labels(active_model, active_loader, device)

print("Computing t-SNE embedding...")
tsne = TSNE(n_components=3, perplexity=30, max_iter=1000,init='pca', random_state=42)

embedded_features = tsne.fit_transform(features)

class_names = ['Resting', 'Math & Story', 'Working Memory', 'Motor']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for class_idx in range(4):
    indices = np.where(labels == class_idx)
    ax.scatter(
        embedded_features[indices, 0], 
        embedded_features[indices, 1], 
        embedded_features[indices, 2],
        label=class_names[class_idx],
        alpha=0.7,
        c=colors[class_idx],
        s=30
    )

plt.title('t-SNE embedding of MEG brain states')
ax.set_xlabel('Dim 1')
ax.set_ylabel('Dim 2')
ax.set_zlabel('Dim 3')
plt.legend(title='Tasks')
plt.grid(True, linestyle=':', alpha=0.5)

output_path_3d = "../reports/figures/CNN-GRU/tsne_3d_cnn_gru.png"
os.makedirs(os.path.dirname(output_path_3d), exist_ok=True)
plt.savefig(output_path_3d, dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_names = ['Resting', 'Math & Story', 'Working Memory', 'Motor']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Define the pairs of axes for each perspective
# (DimX, DimY) pairs: (0,1), (0,2), (1,2)
pairs = [(0, 1), (0, 2), (1, 2)]
titles = ['XY Plane', 'XZ Plane', 'YZ Plane']

for i, (dim1, dim2) in enumerate(pairs):
    ax = axes[i]
    for class_idx in range(4):
        indices = np.where(labels == class_idx)
        ax.scatter(
            embedded_features[indices, dim1], 
            embedded_features[indices, dim2], 
            label=class_names[class_idx] if i == 0 else "", 
            alpha=0.6,
            c=colors[class_idx],
            s=25
        )
    ax.set_title(titles[i])
    ax.set_xlabel(f'Dim {dim1+1}')
    ax.set_ylabel(f'Dim {dim2+1}')
    ax.grid(True, linestyle='--', alpha=0.5)

axes[0].legend(title='Tasks')
plt.tight_layout()
plt.savefig("../reports/figures/CNN-GRU/tsne_2d_cnn_gru.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
learning_rates = [1e-3, 5e-4, 1e-4]
hyper_results = {}

print("testing different learning rates")
for lr in learning_rates:
    print(f"\n--- LR: {lr} ---")
    hyper_model = build_model_1D_GRU_Classifier()
    res_CNN_GRU = train_model(hyper_model, intra_train_loader, intra_test_loader, learning_rate=lr, num_epochs=30)
    hyper_results[str(lr)] = res_CNN_GRU

plt.figure(figsize=(10, 6))
for lr, res_CNN_GRU in hyper_results.items():
    
    if lr == str(learning_rates[0]):
        linestyle = '-'
    elif lr == str(learning_rates[1]):
        linestyle = '--'
    elif lr == str(learning_rates[2]):
        linestyle = ':'
    plt.plot(range(1, len(res_CNN_GRU['test_acc']) + 1), res_CNN_GRU['test_acc'], label=f'LR: {lr}', linestyle=linestyle)

plt.title('Test Accuracy vs Epochs for Different LRs (cnn_gru)')
plt.xlabel('Epochs')
plt.ylabel('Test Accuracy')
plt.legend()
plt.savefig("../reports/figures/CNN-GRU/hyperparam_lr_cnn_gru.png")
plt.show()